# 🧠 GRPO — math reasoning with verifiable reward

Uses `GSM8KRewardSignal` (carried over from v2.0) and the auto-resolved Qwen template. Pick any `DOMAIN_NAME` you want — the notebook bootstraps the YAML if it doesn't exist.

In [ ]:
# ── Environment bootstrap (Colab / Brev / local) ──
# Clones the repo into /content on Colab/Brev so `from app...`
# resolves. On a local checkout the repo already exists and the
# clone is skipped; we just chdir into it.
import os, sys, subprocess, pathlib

REPO_URL  = 'https://github.com/valonys/finetuningtraining.git'
REPO_NAME = 'finetuningtraining'
BRANCH    = 'develop'

def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

def _find_repo_root() -> pathlib.Path | None:
    here = pathlib.Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / 'app' / '__init__.py').exists():
            return candidate
    return None

root = _find_repo_root()
if root is None:
    target = pathlib.Path('/content') if _in_colab() else pathlib.Path.cwd()
    dest = target / REPO_NAME
    if not dest.exists():
        print(f'📥 Cloning {REPO_URL} ({BRANCH}) -> {dest}')
        subprocess.check_call(
            ['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(dest)]
        )
    root = dest

os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print(f'✅ Repo root: {root}')

if _in_colab():
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'transformers>=4.44', 'trl>=0.11', 'peft>=0.12',
        'datasets>=2.20', 'accelerate>=0.33', 'pyyaml', 'pydantic>=2.0',
    ])
    print('✅ Dependencies installed')

In [ ]:

DOMAIN_NAME = 'ai_llm_math'   # ← pick any name; e.g. 'asset_integrity_math', 'legal_reasoning'
BASE_MODEL  = 'Qwen/Qwen2.5-1.5B-Instruct'

from app.config_loader import create_domain_config, domain_config_exists, load_domain_config

if not domain_config_exists(DOMAIN_NAME):
    create_domain_config(
        name=DOMAIN_NAME,
        system_prompt='You are a careful mathematical reasoner. Always show your working and end with the final answer in \\boxed{}.',
        constitution=[
            'Show every step of the calculation.',
            'End the response with the final answer inside \\boxed{}.',
        ],
    )
    print(f'✅ Created configs/domains/{DOMAIN_NAME}.yaml')

config = load_domain_config(DOMAIN_NAME)

In [ ]:
from app.trainers import AgnosticGRPOTrainer
from app.trainers.reward_signals import GSM8KRewardSignal

trainer = AgnosticGRPOTrainer(
    config=config,
    base_model_id=BASE_MODEL,
    hf_dataset_config={
        'repo_id': 'openai/gsm8k',
        'split': 'train',
        'subset': 'main',
        'input_column': 'question',
        'output_column': 'answer',
        'max_samples': 500,
    },
    reward_signal=GSM8KRewardSignal(),
)
result = trainer.train()
result